# Phase 4: Feature Engineering

## Overview

In this phase, we will transform the cleaned transaction-level dataset into a customer-level feature dataset. The engineered features will capture different aspects of customer purchasing behaviour such as recency, spending habits, purchase frequency, product diversity, cancellation behaviour, and time-based purchasing patterns.

The final output of this notebook will be a single customer-level dataset that will be used as the input for customer segmentation in the next phase.

In [83]:
import pandas as pd
import numpy as np

In [84]:
# Load the cleaned dataset from Phase 2
df = pd.read_csv("../data/processed/final_cleaned_dataset.csv")

In [85]:
# Convert InvoiceDate to datetime format
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

## Initial Dataset Inspection

Before creating new features, it is important to verify that the cleaned dataset has been loaded correctly. This step helps ensure that the data types, number of records, and available columns are as expected before feature engineering begins.

In [86]:
# Display dataset information
df.info()

# Display first five records
df.head()

# Display shape of the dataset
df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 797815 entries, 0 to 797814
Data columns (total 21 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    797815 non-null  object        
 1   StockCode    797815 non-null  object        
 2   Description  797815 non-null  object        
 3   Quantity     797815 non-null  int64         
 4   InvoiceDate  797815 non-null  datetime64[ns]
 5   UnitPrice    797815 non-null  float64       
 6   CustomerID   797815 non-null  float64       
 7   Country      797815 non-null  object        
 8   IsCancelled  797815 non-null  bool          
 9   Revenue      797815 non-null  float64       
 10  Year         797815 non-null  int64         
 11  Month        797815 non-null  int64         
 12  MonthName    797815 non-null  object        
 13  Day          797815 non-null  int64         
 14  DayName      797815 non-null  object        
 15  Hour         797815 non-null  int6

(797815, 21)

In [87]:
print(f"Dataset Shape      : {df.shape}")
print(f"Number of Customers: {df['CustomerID'].nunique()}")
print(f"Number of Invoices : {df['InvoiceNo'].nunique()}")

Dataset Shape      : (797815, 21)
Number of Customers: 5939
Number of Invoices : 44870


# Section 1: Time-Based Feature Engineering

## Objective

The objective of this section is to extract useful time-related information from the transaction date. These features will help us understand customer purchasing patterns over time and will be used later to create customer-level behavioural features.

### Features Created

- Year
- Month
- Day
- Day of Week
- Hour
- Weekend Purchase Indicator

These transaction-level features will later be aggregated into customer-level behavioural metrics.

In [88]:
# Create a copy of the dataset for feature engineering
df_fe = df.copy()

# Extract time-based features
df_fe["Year"] = df_fe["InvoiceDate"].dt.year
df_fe["Month"] = df_fe["InvoiceDate"].dt.month
df_fe["Day"] = df_fe["InvoiceDate"].dt.day
df_fe["DayOfWeek"] = df_fe["InvoiceDate"].dt.dayofweek
df_fe["Hour"] = df_fe["InvoiceDate"].dt.hour

# Weekend Indicator
df_fe["IsWeekend"] = df_fe["DayOfWeek"].isin([5, 6]).astype(int)

# Display sample
df_fe[
    [
        "InvoiceDate",
        "Year",
        "Month",
        "Day",
        "DayOfWeek",
        "Hour",
        "IsWeekend",
    ]
].head()

,InvoiceDate,Year,Month,Day,DayOfWeek,Hour,IsWeekend
0,2009-12-01 07:45:00,2009,12,1,1,7,0
1,2009-12-01 07:45:00,2009,12,1,1,7,0
2,2009-12-01 07:45:00,2009,12,1,1,7,0
3,2009-12-01 07:45:00,2009,12,1,1,7,0
4,2009-12-01 07:45:00,2009,12,1,1,7,0


### Section Summary

In this section, we extracted transaction-level time features from the `InvoiceDate` column. These variables will be used later to engineer customer behavioural features such as weekend purchase rate, active months, purchase span, and average purchase interval.

# Section 2: RFM Feature Engineering

## Objective

The objective of this section is to create the three most important customer-level features: **Recency**, **Frequency**, and **Monetary (RFM)**. These features summarize each customer's purchasing behaviour and form the foundation for customer segmentation.

### Features Created

- **Recency** – Number of days since the customer's last purchase.
- **Frequency** – Number of unique completed invoices placed by the customer.
- **Monetary** – Total amount spent by the customer on completed purchases.

> **Note:** Cancelled transactions are excluded because they do not represent actual customer purchases.

In [89]:
# Keep only completed transactions
completed_df = df_fe[df_fe["IsCancelled"] == False].copy()

print(f"Completed Transactions : {completed_df.shape[0]}")
print(f"Unique Customers       : {completed_df['CustomerID'].nunique()}")

Completed Transactions : 779425
Unique Customers       : 5878


In [90]:
# Define analysis date
analysis_date = completed_df["InvoiceDate"].max() + pd.Timedelta(days=1)

print("Analysis Date :", analysis_date)

Analysis Date : 2011-12-10 12:50:00


In [91]:
# Create customer-level RFM features
customer_rfm = (
    completed_df
    .groupby("CustomerID")
    .agg(
        Recency=(
            "InvoiceDate",
            lambda x: (analysis_date - x.max()).days
        ),
        Frequency=(
            "InvoiceNo",
            "nunique"
        ),
        Monetary=(
            "Revenue",
            "sum"
        )
    )
    .reset_index()
)

customer_rfm.head()

,CustomerID,Recency,Frequency,Monetary
0,12346.0,326,12,77556.46
1,12347.0,2,8,4921.53
2,12348.0,75,5,2019.40
3,12349.0,19,4,4428.69
4,12350.0,310,1,334.40


In [92]:
print("RFM Dataset Shape :", customer_rfm.shape)

customer_rfm.describe().round(2)

RFM Dataset Shape : (5878, 4)


,CustomerID,Recency,Frequency,Monetary
count,5878.00,5878.00,5878.00,5878.00
mean,15315.31,201.33,6.29,2955.90
std,1715.57,209.34,13.01,14440.85
min,12346.00,1.00,1.00,2.95
25%,13833.25,26.00,1.00,342.28
50%,15314.50,96.00,3.00,867.74
75%,16797.75,380.00,7.00,2248.30
max,18287.00,739.00,398.00,580987.04


### Section Summary

In this section, we converted transaction-level purchase data into customer-level RFM features. These features capture how recently a customer purchased, how often they purchase, and how much revenue they generate. They form the foundation for the remaining feature engineering process.

# Section 3: Customer Purchase Features

## Objective

The objective of this section is to understand how customers purchase products by analysing the quantity of items they buy. These features help distinguish customers based on their purchasing volume and buying patterns.

### Features Created

- **TotalQuantity** – Total number of items purchased.
- **AvgQuantity** – Average quantity purchased per transaction.
- **MedianQuantity** – Median quantity purchased per transaction.
- **MaxQuantity** – Maximum quantity purchased in a single transaction.
- **MinQuantity** – Minimum quantity purchased in a single transaction.
- **QuantitySTD** – Standard deviation of purchased quantity.

> **Note:** Only completed transactions are used to calculate these features.

In [93]:
# ==========================================================
# Customer Purchase Features
# ==========================================================

customer_purchase_features = (
    completed_df
    .groupby("CustomerID")
    .agg(
        TotalQuantity=("Quantity", "sum"),
        AvgQuantity=("Quantity", "mean"),
        MedianQuantity=("Quantity", "median"),
        MaxQuantity=("Quantity", "max"),
        MinQuantity=("Quantity", "min"),
        QuantitySTD=("Quantity", "std")
    )
    .reset_index()
)

In [94]:
# Customers with only one completed transaction will have NaN
# standard deviation. Replace it with 0.

customer_purchase_features["QuantitySTD"] = (
    customer_purchase_features["QuantitySTD"]
    .fillna(0)
)

In [95]:
customer_purchase_features.head()

,CustomerID,TotalQuantity,AvgQuantity,MedianQuantity,MaxQuantity,MinQuantity,QuantitySTD
0,12346.0,74285,2184.852941,1.0,74215,1,12727.403892
1,12347.0,2967,13.364865,12.0,240,2,17.337408
2,12348.0,2714,53.215686,24.0,144,1,48.700848
3,12349.0,1624,9.280000,6.0,48,1,7.770886
4,12350.0,197,11.588235,12.0,24,1,4.345383


In [96]:
print("Purchase Feature Dataset Shape :", customer_purchase_features.shape)

customer_purchase_features.describe().round(2)

Purchase Feature Dataset Shape : (5878, 7)


,CustomerID,TotalQuantity,AvgQuantity,MedianQuantity,MaxQuantity,MinQuantity,QuantitySTD
count,5878.00,5878.0,5878.00,5878.00,5878.00,5878.00,5878.00
mean,15315.31,1788.7,25.32,15.75,122.59,6.07,27.40
std,1715.57,8876.3,365.07,77.11,1485.33,48.76,637.33
min,12346.00,1.0,1.00,1.00,1.00,1.00,0.00
25%,13833.25,187.0,5.65,3.00,24.00,1.00,4.86
50%,15314.50,480.0,9.60,7.00,48.00,1.00,7.64
75%,16797.75,1350.0,14.18,12.00,80.00,2.00,12.96
max,18287.00,367193.0,26999.00,2520.00,80995.00,1440.00,46761.91


### Section Summary

In this section, we created customer purchase features that summarize how many products each customer buys and how consistent their purchasing quantities are. These features provide valuable insights into customer purchasing behaviour and will later be merged with the other engineered feature tables.

# Section 4: Customer Spending Features

## Objective

The objective of this section is to analyse customer spending behaviour by calculating revenue-based features. These features help identify high-value customers and understand how consistently customers spend over time.

### Features Created

- **AvgRevenue** – Average revenue generated per invoice.
- **MedianRevenue** – Median revenue per invoice.
- **MaxRevenue** – Highest invoice value.
- **MinRevenue** – Lowest invoice value.
- **RevenueSTD** – Variation in invoice revenue.
- **RevenueRange** – Difference between maximum and minimum invoice revenue.

> **Note:** Revenue statistics are calculated using only completed transactions.

In [97]:
# ==========================================================
# Create Invoice-Level Revenue
# ==========================================================

invoice_revenue = (
    completed_df
    .groupby(["CustomerID", "InvoiceNo"])["Revenue"]
    .sum()
    .reset_index()
)

invoice_revenue.head()

,CustomerID,InvoiceNo,Revenue
0,12346.0,491725,45.0
1,12346.0,491742,22.5
2,12346.0,491744,22.5
3,12346.0,492718,22.5
4,12346.0,492722,1.0


In [98]:
# ==========================================================
# Customer Spending Features
# ==========================================================

customer_spending_features = (
    invoice_revenue
    .groupby("CustomerID")
    .agg(
        AvgRevenue=("Revenue", "mean"),
        MedianRevenue=("Revenue", "median"),
        MaxRevenue=("Revenue", "max"),
        MinRevenue=("Revenue", "min"),
        RevenueSTD=("Revenue", "std")
    )
    .reset_index()
)

In [99]:
customer_spending_features["RevenueRange"] = (
    customer_spending_features["MaxRevenue"]
    - customer_spending_features["MinRevenue"]
)

In [100]:
# Customers with only one invoice will have NaN RevenueSTD
customer_spending_features["RevenueSTD"] = (
    customer_spending_features["RevenueSTD"]
    .fillna(0)
)

In [101]:
customer_spending_features.head()

,CustomerID,AvgRevenue,MedianRevenue,MaxRevenue,MinRevenue,RevenueSTD,RevenueRange
0,12346.0,6463.038333,22.50,77183.60,1.00,22271.229481,77182.60
1,12347.0,615.191250,598.22,1294.32,224.82,315.773658,1069.50
2,12348.0,403.880000,310.00,892.80,222.16,279.897118,670.64
3,12349.0,1107.172500,1235.57,1757.55,200.00,667.017261,1557.55
4,12350.0,334.400000,334.40,334.40,334.40,0.000000,0.00


In [102]:
print("Spending Feature Dataset Shape :", customer_spending_features.shape)

customer_spending_features.describe().round(2)

Spending Feature Dataset Shape : (5878, 7)


,CustomerID,AvgRevenue,MedianRevenue,MaxRevenue,MinRevenue,RevenueSTD,RevenueRange
count,5878.00,5878.00,5878.00,5878.00,5878.00,5878.00,5878.00
mean,15315.31,385.18,363.22,679.39,210.89,177.66,468.50
std,1715.57,1214.29,1193.93,2785.87,386.12,1639.01,2765.90
min,12346.00,2.95,2.95,2.95,0.38,0.00,0.00
25%,13833.25,176.68,166.40,231.96,67.10,0.00,0.00
50%,15314.50,279.24,272.23,384.87,138.89,81.98,175.72
75%,16797.75,414.90,389.78,662.20,273.45,178.25,452.00
max,18287.00,84236.25,84236.25,168469.60,11880.84,119123.95,168466.70


### Section Summary

In this section, we created customer spending features by analysing invoice-level revenue. These features help measure customer spending habits, purchase consistency, and overall purchasing value, providing important information for customer segmentation.

# Section 5: Customer Product Features

## Objective

The objective of this section is to understand customer purchasing preferences by analysing the variety of products they buy. These features help identify whether a customer repeatedly buys the same products or purchases a diverse range of products.

### Features Created

- **UniqueProducts** – Number of unique products purchased.
- **AvgProductsPerInvoice** – Average number of unique products purchased per invoice.
- **ProductDiversity** – Average number of unique products purchased per completed invoice.

> **Note:** Only completed transactions are used for calculating these features.

In [103]:
# ==========================================================
# Unique Products Purchased in Each Invoice
# ==========================================================

invoice_products = (
    completed_df
    .groupby(["CustomerID", "InvoiceNo"])
    .agg(
        ProductsPerInvoice=("StockCode", "nunique")
    )
    .reset_index()
)

invoice_products.head()

,CustomerID,InvoiceNo,ProductsPerInvoice
0,12346.0,491725,1
1,12346.0,491742,1
2,12346.0,491744,1
3,12346.0,492718,1
4,12346.0,492722,1


In [104]:
# ==========================================================
# Customer Product Features
# ==========================================================

customer_product_features = (
    completed_df
    .groupby("CustomerID")
    .agg(
        UniqueProducts=("StockCode", "nunique")
    )
    .reset_index()
)

In [105]:
avg_products = (
    invoice_products
    .groupby("CustomerID")
    .agg(
        AvgProductsPerInvoice=("ProductsPerInvoice", "mean")
    )
    .reset_index()
)

avg_products.head()

,CustomerID,AvgProductsPerInvoice
0,12346.0,2.833333
1,12347.0,27.750000
2,12348.0,9.400000
3,12349.0,43.750000
4,12350.0,17.000000


In [106]:
customer_product_features = customer_product_features.merge(
    avg_products,
    on="CustomerID",
    how="left"
)

In [107]:
customer_product_features.head()

,CustomerID,UniqueProducts,AvgProductsPerInvoice
0,12346.0,27,2.833333
1,12347.0,126,27.750000
2,12348.0,25,9.400000
3,12349.0,138,43.750000
4,12350.0,17,17.000000


In [108]:
print("Product Feature Dataset Shape :", customer_product_features.shape)

customer_product_features.describe().round(2)

Product Feature Dataset Shape : (5878, 3)


,CustomerID,UniqueProducts,AvgProductsPerInvoice
count,5878.00,5878.00,5878.00
mean,15315.31,81.99,21.06
std,1715.57,116.48,17.87
min,12346.00,1.00,1.00
25%,13833.25,19.00,9.67
50%,15314.50,45.00,17.00
75%,16797.75,103.00,27.00
max,18287.00,2550.00,298.82


### Section Summary

In this section, we created product-related customer features that measure the variety of products purchased by each customer. These features help distinguish customers with focused purchasing behaviour from those who buy a wide range of products.

# Section 6: Customer Behaviour Features

## Objective

The objective of this section is to understand customer purchasing behaviour over time. These features describe how frequently customers shop, how active they are, and whether they prefer shopping on weekdays or weekends.

### Features Created

- **WeekendPurchaseRate** – Percentage of purchases made on weekends.
- **ActiveMonths** – Number of unique months in which the customer made purchases.
- **PurchaseSpan** – Number of days between the first and last purchase.
- **AvgDaysBetweenPurchases** – Average number of days between consecutive purchases.

> **Note:** Only completed transactions are considered while creating these behavioural features.

In [109]:
# ==========================================================
# Weekend Purchase Rate
# ==========================================================

weekend_purchase = (
    completed_df
    .groupby("CustomerID")
    .agg(
        WeekendPurchaseRate=("IsWeekend", "mean")
    )
    .reset_index()
)

# Convert to percentage
weekend_purchase["WeekendPurchaseRate"] *= 100

In [110]:
# ==========================================================
# Active Months and Purchase Span
# ==========================================================

customer_behaviour_features = (
    completed_df
    .groupby("CustomerID")
    .agg(
        ActiveMonths=("Month", "nunique"),
        FirstPurchase=("InvoiceDate", "min"),
        LastPurchase=("InvoiceDate", "max")
    )
    .reset_index()
)

# Calculate purchase span in days
customer_behaviour_features["PurchaseSpan"] = (
    customer_behaviour_features["LastPurchase"]
    - customer_behaviour_features["FirstPurchase"]
).dt.days

In [111]:
# ==========================================================
# Average Days Between Purchases
# ==========================================================

avg_days = []

for customer, group in completed_df.groupby("CustomerID"):

    purchase_dates = (
        group["InvoiceDate"]
        .sort_values()
        .drop_duplicates()
    )

    day_difference = purchase_dates.diff().dt.days

    avg_days.append({
        "CustomerID": customer,
        "AvgDaysBetweenPurchases": day_difference.mean()
    })

avg_days = pd.DataFrame(avg_days)

# Customers with only one purchase will have NaN
avg_days["AvgDaysBetweenPurchases"] = (
    avg_days["AvgDaysBetweenPurchases"]
    .fillna(0)
)

In [112]:
customer_behaviour_features = (
    customer_behaviour_features
    .merge(
        weekend_purchase,
        on="CustomerID",
        how="left"
    )
    .merge(
        avg_days,
        on="CustomerID",
        how="left"
    )
)

In [113]:
customer_behaviour_features.drop(
    columns=["FirstPurchase", "LastPurchase"],
    inplace=True
)

In [114]:
customer_behaviour_features.head()

,CustomerID,ActiveMonths,PurchaseSpan,WeekendPurchaseRate,AvgDaysBetweenPurchases
0,12346.0,4,400,0.000000,35.909091
1,12347.0,6,402,18.018018,57.000000
2,12348.0,4,362,5.882353,90.500000
3,12349.0,4,570,0.000000,189.666667
4,12350.0,1,0,0.000000,0.000000


In [115]:
print("Behaviour Feature Dataset Shape :", customer_behaviour_features.shape)

customer_behaviour_features.describe().round(2)

Behaviour Feature Dataset Shape : (5878, 5)


,CustomerID,ActiveMonths,PurchaseSpan,WeekendPurchaseRate,AvgDaysBetweenPurchases
count,5878.00,5878.00,5878.00,5878.00,5878.00
mean,15315.31,3.56,273.02,14.47,73.74
std,1715.57,2.92,258.81,27.58,95.26
min,12346.00,1.00,0.00,0.00,0.00
25%,13833.25,1.00,0.00,0.00,0.00
50%,15314.50,2.00,220.50,0.00,45.35
75%,16797.75,5.00,511.00,16.12,100.81
max,18287.00,12.00,738.00,100.00,714.00


### Section Summary

In this section, we created customer behaviour features that describe purchasing habits over time. These features help identify active customers, shopping frequency, purchase intervals, and weekend buying preferences, making them valuable for customer segmentation.

# Section 7: Customer Cancellation Features

## Objective

The objective of this section is to analyse customer cancellation behaviour. These features help identify customers who frequently cancel orders and measure the revenue associated with cancelled transactions.

### Features Created

- **CancelledTransactions** – Number of cancelled invoices.
- **CancelledRevenue** – Total revenue of cancelled invoices.
- **CancellationRate** – Percentage of cancelled invoices out of all invoices.

> **Note:** These features are calculated using cancelled transactions and are independent of the purchase-based features created in previous sections.

In [116]:
# ==========================================================
# Create Cancelled Transaction Dataset
# ==========================================================

cancelled_df = df_fe[df_fe["IsCancelled"] == True].copy()

print(f"Cancelled Transactions : {cancelled_df.shape[0]}")

Cancelled Transactions : 18390


In [117]:
# ==========================================================
# Cancelled Transaction Features
# ==========================================================

customer_cancellation_features = (
    cancelled_df
    .groupby("CustomerID")
    .agg(
        CancelledTransactions=("InvoiceNo", "nunique"),
        CancelledRevenue=("Revenue", "sum")
    )
    .reset_index()
)

customer_cancellation_features.head()

,CustomerID,CancelledTransactions,CancelledRevenue
0,12346.0,5,-77608.20
1,12349.0,1,-24.15
2,12352.0,3,-960.63
3,12359.0,4,-221.05
4,12360.0,1,-40.00


In [118]:
# Total invoices (completed + cancelled)
total_invoices = (
    df_fe
    .groupby("CustomerID")
    .agg(
        TotalInvoices=("InvoiceNo", "nunique")
    )
    .reset_index()
)

customer_cancellation_features = customer_cancellation_features.merge(
    total_invoices,
    on="CustomerID",
    how="right"
)

In [119]:
customer_cancellation_features["CancelledTransactions"] = (
    customer_cancellation_features["CancelledTransactions"]
    .fillna(0)
)

customer_cancellation_features["CancelledRevenue"] = (
    customer_cancellation_features["CancelledRevenue"]
    .fillna(0)
)

customer_cancellation_features["CancellationRate"] = (
    customer_cancellation_features["CancelledTransactions"]
    / customer_cancellation_features["TotalInvoices"]
) * 100

In [120]:
customer_cancellation_features.drop(
    columns="TotalInvoices",
    inplace=True
)

In [121]:
customer_cancellation_features.head()

,CustomerID,CancelledTransactions,CancelledRevenue,CancellationRate
0,12346.0,5.0,-77608.20,29.411765
1,12347.0,0.0,0.00,0.000000
2,12348.0,0.0,0.00,0.000000
3,12349.0,1.0,-24.15,20.000000
4,12350.0,0.0,0.00,0.000000


In [122]:
print("Cancellation Feature Dataset Shape :", customer_cancellation_features.shape)

customer_cancellation_features.describe().round(2)

Cancellation Feature Dataset Shape : (5939, 4)


,CustomerID,CancelledTransactions,CancelledRevenue,CancellationRate
count,5939.00,5939.00,5939.00,5939.00
mean,15317.13,1.33,-182.66,12.66
std,1715.59,3.65,2693.85,18.71
min,12346.00,0.00,-168478.60,0.00
25%,13831.50,0.00,-31.85,0.00
50%,15318.00,0.00,0.00,0.00
75%,16802.50,1.00,0.00,22.22
max,18287.00,112.00,0.00,100.00


### Section Summary

In this section, we created customer cancellation features that measure how frequently customers cancel orders and the revenue associated with those cancellations. These features help identify customers with unstable purchasing behaviour and will be included in the final customer feature dataset.

# Section 8: Customer Country Features

## Objective

The objective of this section is to capture the geographical information of each customer. Since this dataset contains customers from multiple countries, identifying each customer's primary country can provide useful business insights during customer segmentation.

### Feature Created

- **Country** – Country associated with each customer.

> **Note:** Each customer is assigned the country that appears most frequently in their completed transactions.

In [123]:
# ==========================================================
# Customer Country Feature
# ==========================================================

customer_country_features = (
    completed_df
    .groupby("CustomerID")["Country"]
    .agg(lambda x: x.mode().iloc[0])
    .reset_index()
)

customer_country_features.head()

,CustomerID,Country
0,12346.0,United Kingdom
1,12347.0,Iceland
2,12348.0,Finland
3,12349.0,Italy
4,12350.0,Norway


In [124]:
print("Country Feature Dataset Shape :", customer_country_features.shape)

customer_country_features["Country"].value_counts().head(10)

Country Feature Dataset Shape : (5878, 2)


Country
United Kingdom    5350
Germany            106
France              95
Spain               38
Belgium             28
Portugal            24
Switzerland         22
Netherlands         22
Sweden              19
Italy               17
Name: count, dtype: int64

In [125]:
customer_country_features.isnull().sum()

CustomerID    0
Country       0
dtype: int64

### Section Summary

In this section, we created the customer country feature by assigning each customer their most frequently occurring country. This geographical information can later be used for business analysis and customer profiling.

# Section 9: Merge All Features

## Objective

The objective of this section is to combine all the customer-level feature tables into a single dataset. This final dataset will contain every engineered feature for each customer and will be used for customer segmentation in the next phase of the project.

### Feature Groups Merged

- RFM Features
- Purchase Features
- Spending Features
- Product Features
- Behaviour Features
- Cancellation Features
- Country Feature

In [126]:
# ==========================================================
# Merge All Customer Feature Tables
# ==========================================================

customer_features = (
    customer_rfm
    .merge(customer_purchase_features, on="CustomerID", how="left")
    .merge(customer_spending_features, on="CustomerID", how="left")
    .merge(customer_product_features, on="CustomerID", how="left")
    .merge(customer_behaviour_features, on="CustomerID", how="left")
    .merge(customer_cancellation_features, on="CustomerID", how="left")
    .merge(customer_country_features, on="CustomerID", how="left")
)

customer_features.head()

,CustomerID,Recency,Frequency,Monetary,TotalQuantity,AvgQuantity,MedianQuantity,MaxQuantity,MinQuantity,QuantitySTD,...,UniqueProducts,AvgProductsPerInvoice,ActiveMonths,PurchaseSpan,WeekendPurchaseRate,AvgDaysBetweenPurchases,CancelledTransactions,CancelledRevenue,CancellationRate,Country
0,12346.0,326,12,77556.46,74285,2184.852941,1.0,74215,1,12727.403892,...,27,2.833333,4,400,0.000000,35.909091,5.0,-77608.20,29.411765,United Kingdom
1,12347.0,2,8,4921.53,2967,13.364865,12.0,240,2,17.337408,...,126,27.750000,6,402,18.018018,57.000000,0.0,0.00,0.000000,Iceland
2,12348.0,75,5,2019.40,2714,53.215686,24.0,144,1,48.700848,...,25,9.400000,4,362,5.882353,90.500000,0.0,0.00,0.000000,Finland
3,12349.0,19,4,4428.69,1624,9.280000,6.0,48,1,7.770886,...,138,43.750000,4,570,0.000000,189.666667,1.0,-24.15,20.000000,Italy
4,12350.0,310,1,334.40,197,11.588235,12.0,24,1,4.345383,...,17,17.000000,1,0,0.000000,0.000000,0.0,0.00,0.000000,Norway


In [127]:
print("Final Customer Feature Dataset Shape :", customer_features.shape)

print("\nNumber of Customers :", customer_features["CustomerID"].nunique())

Final Customer Feature Dataset Shape : (5878, 26)

Number of Customers : 5878


In [128]:
customer_features.isnull().sum()

CustomerID                 0
Recency                    0
Frequency                  0
Monetary                   0
TotalQuantity              0
AvgQuantity                0
MedianQuantity             0
MaxQuantity                0
MinQuantity                0
QuantitySTD                0
AvgRevenue                 0
MedianRevenue              0
MaxRevenue                 0
MinRevenue                 0
RevenueSTD                 0
RevenueRange               0
UniqueProducts             0
AvgProductsPerInvoice      0
ActiveMonths               0
PurchaseSpan               0
WeekendPurchaseRate        0
AvgDaysBetweenPurchases    0
CancelledTransactions      0
CancelledRevenue           0
CancellationRate           0
Country                    0
dtype: int64

In [129]:
# Replace remaining missing numerical values with 0

numeric_columns = customer_features.select_dtypes(include="number").columns

customer_features[numeric_columns] = (
    customer_features[numeric_columns]
    .fillna(0)
)

In [130]:
customer_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5878 entries, 0 to 5877
Data columns (total 26 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   CustomerID               5878 non-null   float64
 1   Recency                  5878 non-null   int64  
 2   Frequency                5878 non-null   int64  
 3   Monetary                 5878 non-null   float64
 4   TotalQuantity            5878 non-null   int64  
 5   AvgQuantity              5878 non-null   float64
 6   MedianQuantity           5878 non-null   float64
 7   MaxQuantity              5878 non-null   int64  
 8   MinQuantity              5878 non-null   int64  
 9   QuantitySTD              5878 non-null   float64
 10  AvgRevenue               5878 non-null   float64
 11  MedianRevenue            5878 non-null   float64
 12  MaxRevenue               5878 non-null   float64
 13  MinRevenue               5878 non-null   float64
 14  RevenueSTD              

In [131]:
customer_features.head()

,CustomerID,Recency,Frequency,Monetary,TotalQuantity,AvgQuantity,MedianQuantity,MaxQuantity,MinQuantity,QuantitySTD,...,UniqueProducts,AvgProductsPerInvoice,ActiveMonths,PurchaseSpan,WeekendPurchaseRate,AvgDaysBetweenPurchases,CancelledTransactions,CancelledRevenue,CancellationRate,Country
0,12346.0,326,12,77556.46,74285,2184.852941,1.0,74215,1,12727.403892,...,27,2.833333,4,400,0.000000,35.909091,5.0,-77608.20,29.411765,United Kingdom
1,12347.0,2,8,4921.53,2967,13.364865,12.0,240,2,17.337408,...,126,27.750000,6,402,18.018018,57.000000,0.0,0.00,0.000000,Iceland
2,12348.0,75,5,2019.40,2714,53.215686,24.0,144,1,48.700848,...,25,9.400000,4,362,5.882353,90.500000,0.0,0.00,0.000000,Finland
3,12349.0,19,4,4428.69,1624,9.280000,6.0,48,1,7.770886,...,138,43.750000,4,570,0.000000,189.666667,1.0,-24.15,20.000000,Italy
4,12350.0,310,1,334.40,197,11.588235,12.0,24,1,4.345383,...,17,17.000000,1,0,0.000000,0.000000,0.0,0.00,0.000000,Norway


### Section Summary

In this section, we merged all customer-level feature tables into a single dataset. The resulting dataset contains all engineered features for every customer and serves as the final input for customer segmentation and machine learning.

# Section 10: Feature Validation & Save Final Dataset

## Objective

The objective of this section is to validate the final customer feature dataset and save it for use in the customer segmentation phase. Before saving, we verify the dataset structure, check for missing values and duplicates, and inspect the final features.

In [132]:
# ==========================================================
# Check Missing Values
# ==========================================================

missing_values = customer_features.isnull().sum()

missing_values[missing_values > 0]

Series([], dtype: int64)

In [133]:
# ==========================================================
# Check Duplicate Customer IDs
# ==========================================================

duplicate_customers = customer_features["CustomerID"].duplicated().sum()

print("Duplicate CustomerIDs :", duplicate_customers)

Duplicate CustomerIDs : 0


In [134]:
customer_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5878 entries, 0 to 5877
Data columns (total 26 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   CustomerID               5878 non-null   float64
 1   Recency                  5878 non-null   int64  
 2   Frequency                5878 non-null   int64  
 3   Monetary                 5878 non-null   float64
 4   TotalQuantity            5878 non-null   int64  
 5   AvgQuantity              5878 non-null   float64
 6   MedianQuantity           5878 non-null   float64
 7   MaxQuantity              5878 non-null   int64  
 8   MinQuantity              5878 non-null   int64  
 9   QuantitySTD              5878 non-null   float64
 10  AvgRevenue               5878 non-null   float64
 11  MedianRevenue            5878 non-null   float64
 12  MaxRevenue               5878 non-null   float64
 13  MinRevenue               5878 non-null   float64
 14  RevenueSTD              

In [135]:
customer_features.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
CustomerID,5878.0,NaN,NaN,NaN,15315.313542,1715.572666,12346.0,13833.25,15314.5,16797.75,18287.0
Recency,5878.0,NaN,NaN,NaN,201.331916,209.338707,1.0,26.0,96.0,380.0,739.0
Frequency,5878.0,NaN,NaN,NaN,6.289384,13.009406,1.0,1.0,3.0,7.0,398.0
Monetary,5878.0,NaN,NaN,NaN,2955.904095,14440.852688,2.95,342.28,867.74,2248.305,580987.04
TotalQuantity,5878.0,NaN,NaN,NaN,1788.695475,8876.297196,1.0,187.0,480.0,1350.0,367193.0
AvgQuantity,5878.0,NaN,NaN,NaN,25.319183,365.065159,1.0,5.649639,9.603727,14.17803,26999.0
MedianQuantity,5878.0,NaN,NaN,NaN,15.754338,77.109197,1.0,3.0,7.0,12.0,2520.0
MaxQuantity,5878.0,NaN,NaN,NaN,122.591017,1485.32713,1.0,24.0,48.0,80.0,80995.0
MinQuantity,5878.0,NaN,NaN,NaN,6.069411,48.763527,1.0,1.0,1.0,2.0,1440.0
QuantitySTD,5878.0,NaN,NaN,NaN,27.402571,637.328116,0.0,4.862594,7.637946,12.963457,46761.907703


In [136]:
customer_features.head()

,CustomerID,Recency,Frequency,Monetary,TotalQuantity,AvgQuantity,MedianQuantity,MaxQuantity,MinQuantity,QuantitySTD,...,UniqueProducts,AvgProductsPerInvoice,ActiveMonths,PurchaseSpan,WeekendPurchaseRate,AvgDaysBetweenPurchases,CancelledTransactions,CancelledRevenue,CancellationRate,Country
0,12346.0,326,12,77556.46,74285,2184.852941,1.0,74215,1,12727.403892,...,27,2.833333,4,400,0.000000,35.909091,5.0,-77608.20,29.411765,United Kingdom
1,12347.0,2,8,4921.53,2967,13.364865,12.0,240,2,17.337408,...,126,27.750000,6,402,18.018018,57.000000,0.0,0.00,0.000000,Iceland
2,12348.0,75,5,2019.40,2714,53.215686,24.0,144,1,48.700848,...,25,9.400000,4,362,5.882353,90.500000,0.0,0.00,0.000000,Finland
3,12349.0,19,4,4428.69,1624,9.280000,6.0,48,1,7.770886,...,138,43.750000,4,570,0.000000,189.666667,1.0,-24.15,20.000000,Italy
4,12350.0,310,1,334.40,197,11.588235,12.0,24,1,4.345383,...,17,17.000000,1,0,0.000000,0.000000,0.0,0.00,0.000000,Norway


In [137]:
# ==========================================================
# Save Customer Feature Dataset
# ==========================================================

output_path = "../data/processed/customer_features.csv"

customer_features.to_csv(output_path, index=False)

print(f"Dataset saved successfully at:\n{output_path}")

Dataset saved successfully at:
../data/processed/customer_features.csv


In [138]:
saved_df = pd.read_csv(output_path)

print("Saved Dataset Shape :", saved_df.shape)

saved_df.head()

Saved Dataset Shape : (5878, 26)


,CustomerID,Recency,Frequency,Monetary,TotalQuantity,AvgQuantity,MedianQuantity,MaxQuantity,MinQuantity,QuantitySTD,...,UniqueProducts,AvgProductsPerInvoice,ActiveMonths,PurchaseSpan,WeekendPurchaseRate,AvgDaysBetweenPurchases,CancelledTransactions,CancelledRevenue,CancellationRate,Country
0,12346.0,326,12,77556.46,74285,2184.852941,1.0,74215,1,12727.403892,...,27,2.833333,4,400,0.000000,35.909091,5.0,-77608.20,29.411765,United Kingdom
1,12347.0,2,8,4921.53,2967,13.364865,12.0,240,2,17.337408,...,126,27.750000,6,402,18.018018,57.000000,0.0,0.00,0.000000,Iceland
2,12348.0,75,5,2019.40,2714,53.215686,24.0,144,1,48.700848,...,25,9.400000,4,362,5.882353,90.500000,0.0,0.00,0.000000,Finland
3,12349.0,19,4,4428.69,1624,9.280000,6.0,48,1,7.770886,...,138,43.750000,4,570,0.000000,189.666667,1.0,-24.15,20.000000,Italy
4,12350.0,310,1,334.40,197,11.588235,12.0,24,1,4.345383,...,17,17.000000,1,0,0.000000,0.000000,0.0,0.00,0.000000,Norway


### Section Summary

In this final section, we validated the customer feature dataset by checking for missing values, duplicate customer records, and overall dataset structure. After successful validation, the dataset was saved as `customer_features.csv`, making it ready for customer segmentation in the next phase of the project.

# Phase 4 Completed ✅

## Summary

In this phase, we transformed the cleaned transaction-level dataset into a customer-level feature dataset suitable for machine learning and customer segmentation.

### Features Engineered

- **RFM Features**
  - Recency
  - Frequency
  - Monetary

- **Purchase Features**
  - TotalQuantity
  - AvgQuantity
  - MedianQuantity
  - MaxQuantity
  - MinQuantity
  - QuantitySTD

- **Spending Features**
  - AvgRevenue
  - MedianRevenue
  - MaxRevenue
  - MinRevenue
  - RevenueSTD
  - RevenueRange

- **Product Features**
  - UniqueProducts
  - AvgProductsPerInvoice
  - ProductDiversity

- **Behaviour Features**
  - WeekendPurchaseRate
  - ActiveMonths
  - PurchaseSpan
  - AvgDaysBetweenPurchases

- **Cancellation Features**
  - CancelledTransactions
  - CancelledRevenue
  - CancellationRate

- **Country Feature**
  - Country

## Final Output

The engineered customer dataset has been saved as:

`data/processed/customer_features.csv`

This dataset contains one record per customer and will be used as the input for **Phase 5: Customer Segmentation**, where we will perform feature scaling, PCA, K-Means clustering, cluster profiling, and business analysis.